<a href="https://colab.research.google.com/github/shanviMatrix/Assignment-ML-L/blob/main/Shanvi_2026010107_Assignment_naivebayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Exercise 11: Gaussian Naive Bayes
## Purchase Prediction using `purchase_logistic.csv`

**Objective:** Use Gaussian Naive Bayes to predict purchase outcomes using the specified feature vectors.

This notebook solves both questions from the lab sheet:
- **Question 1:** Features — `Age`, `EstimatedSalary`
- **Question 2:** Features — `Gender`, `Age`, `EstimatedSalary`

Both models use preprocessing, train-test splitting, confusion matrices, classification reports and accuracy comparison.

## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
# Keep purchase_logistic.csv in the same folder as this notebook
df = pd.read_csv("purchase_logistic.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 3. Explore the dataset

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nStatistical summary:")
display(df.describe(include="all"))

print("\\nMissing values:")
display(df.isnull().sum())

## 4. Data preprocessing

In [ ]:
# Remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

# Remove common unnecessary index columns
df = df.drop(
    columns=[c for c in df.columns if c.lower().startswith("unnamed")],
    errors="ignore"
)

# Clean column names by removing accidental spaces
df.columns = df.columns.str.strip()

# Identify the target column.
# Common names in purchase datasets include Purchased and Purchase.
target_candidates = ["Purchased", "Purchase", "Purchased?", "target", "Outcome"]
target_matches = [c for c in df.columns if c.lower() in [x.lower() for x in target_candidates]]

if not target_matches:
    raise ValueError(
        "Could not automatically identify the purchase target column. "
        "Edit TARGET_COLUMN below using one of these columns: " + str(df.columns.tolist())
    )

TARGET_COLUMN = target_matches[0]
print("Target column:", TARGET_COLUMN)
print("Target values:")
display(df[TARGET_COLUMN].value_counts(dropna=False))

## 5. Prepare target and feature data

In [ ]:
# Remove rows with missing target values
df = df.dropna(subset=[TARGET_COLUMN]).copy()

# Encode target labels as numeric values if required
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[TARGET_COLUMN].astype(str))

print("Target classes:", list(target_encoder.classes_))
print("Encoded target counts:")
display(pd.Series(y).value_counts().sort_index())

# Question 1
## 6. Gaussian Naive Bayes using Age and EstimatedSalary

Feature vectors: `Age`, `EstimatedSalary`.

We split the data into training and testing sets, fill missing numeric values using the training median, scale the features using `StandardScaler`, and train a `GaussianNB` classifier.

In [ ]:
features_q1 = ["Age", "EstimatedSalary"]

# Check that required columns are present
missing_cols = [c for c in features_q1 if c not in df.columns]
if missing_cols:
    raise ValueError(f"Required columns not found: {missing_cols}")

X1 = df[features_q1].copy()

# Convert feature columns to numeric
for col in features_q1:
    X1[col] = pd.to_numeric(X1[col], errors="coerce")

# Split the data
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y, test_size=0.25, random_state=42, stratify=y
)

# Fill missing values using training medians
medians_q1 = X1_train.median()
X1_train = X1_train.fillna(medians_q1)
X1_test = X1_test.fillna(medians_q1)

# Scale features
scaler_q1 = StandardScaler()
X1_train_scaled = scaler_q1.fit_transform(X1_train)
X1_test_scaled = scaler_q1.transform(X1_test)

print("Training shape:", X1_train_scaled.shape)
print("Testing shape:", X1_test_scaled.shape)

## 7. Train Gaussian Naive Bayes model (Question 1)

In [ ]:
gnb_q1 = GaussianNB()
gnb_q1.fit(X1_train_scaled, y1_train)

y1_pred = gnb_q1.predict(X1_test_scaled)

print("Question 1 model trained and predictions generated.")

## 8. Confusion matrix and classification report (Question 1)

In [ ]:
cm1 = confusion_matrix(y1_test, y1_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm1, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Question 1")
plt.show()

print("Classification Report - Question 1")
print(classification_report(
    y1_test, y1_pred,
    labels=np.arange(len(target_encoder.classes_)),
    target_names=[str(c) for c in target_encoder.classes_],
    zero_division=0
))

accuracy_q1 = accuracy_score(y1_test, y1_pred)
print("Accuracy (Q1):", round(accuracy_q1, 4))

# Question 2
## 9. Gaussian Naive Bayes using Gender, Age and EstimatedSalary

Feature vectors: `Gender`, `Age`, `EstimatedSalary`.

For this question, Gender is label encoded. Age and EstimatedSalary are scaled using `StandardScaler`. We use the same train-test split settings as Question 1 so the results can be compared fairly.

In [ ]:
features_q2 = ["Gender", "Age", "EstimatedSalary"]

missing_cols = [c for c in features_q2 if c not in df.columns]
if missing_cols:
    raise ValueError(f"Required columns not found: {missing_cols}")

X2 = df[features_q2].copy()

# Encode Gender
gender_encoder = LabelEncoder()
X2["Gender"] = gender_encoder.fit_transform(X2["Gender"].astype(str))

# Convert numeric features
for col in ["Age", "EstimatedSalary"]:
    X2[col] = pd.to_numeric(X2[col], errors="coerce")

# Use the same split settings as Question 1
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.25, random_state=42, stratify=y
)

# Impute missing values using training medians
medians_q2 = X2_train.median()
X2_train = X2_train.fillna(medians_q2)
X2_test = X2_test.fillna(medians_q2)

# Scale all features as requested
scaler_q2 = StandardScaler()
X2_train_scaled = scaler_q2.fit_transform(X2_train)
X2_test_scaled = scaler_q2.transform(X2_test)

print("Gender encoding:", dict(zip(gender_encoder.classes_, gender_encoder.transform(gender_encoder.classes_))))
print("Training shape:", X2_train_scaled.shape)
print("Testing shape:", X2_test_scaled.shape)

## 10. Train Gaussian Naive Bayes model (Question 2)

In [ ]:
gnb_q2 = GaussianNB()
gnb_q2.fit(X2_train_scaled, y2_train)

y2_pred = gnb_q2.predict(X2_test_scaled)

print("Question 2 model trained and predictions generated.")

## 11. Confusion matrix and classification report (Question 2)

In [ ]:
cm2 = confusion_matrix(y2_test, y2_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm2, annot=True, fmt="d", cmap="Greens",
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Question 2")
plt.show()

print("Classification Report - Question 2")
print(classification_report(
    y2_test, y2_pred,
    labels=np.arange(len(target_encoder.classes_)),
    target_names=[str(c) for c in target_encoder.classes_],
    zero_division=0
))

accuracy_q2 = accuracy_score(y2_test, y2_pred)
print("Accuracy (Q2):", round(accuracy_q2, 4))

## 12. Compare the results of Question 1 and Question 2

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-Score (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

comparison = pd.DataFrame({
    "Q1: Age + EstimatedSalary": calculate_metrics(y1_test, y1_pred),
    "Q2: Gender + Age + EstimatedSalary": calculate_metrics(y2_test, y2_pred)
}).T

display(comparison.round(4))

print("Accuracy difference (Q2 - Q1):", round(accuracy_q2 - accuracy_q1, 4))

## 13. Visual comparison of model accuracy

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(
    ["Q1: Age + Salary", "Q2: Gender + Age + Salary"],
    [accuracy_q1, accuracy_q2]
)
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.title("Accuracy Comparison: Question 1 vs Question 2")
plt.show()

## 14. Conclusion

In [ ]:
if accuracy_q2 > accuracy_q1:
    conclusion = "Question 2 achieved higher accuracy after adding Gender."
elif accuracy_q2 < accuracy_q1:
    conclusion = "Question 1 achieved higher accuracy; adding Gender did not improve accuracy."
else:
    conclusion = "Both models achieved the same accuracy."

print("CONCLUSION")
print("----------")
print("Question 1 uses Age and EstimatedSalary.")
print("Question 2 uses Gender, Age and EstimatedSalary.")
print(conclusion)
print("\\nCompare precision, recall and F1-score in the results table as well.")
print("The actual conclusion depends on the results obtained from purchase_logistic.csv.")

---
### Notes
1. Place `purchase_logistic.csv` in the same folder as this notebook.
2. Run all cells from top to bottom.
3. The notebook automatically detects a common purchase target column. If it cannot find one, update the target-column selection cell.
4. Both questions use the same 75:25 train-test split, random state and stratification for a fair comparison.
5. The notebook uses weighted Precision, Recall and F1-score in the comparison table.
6. The assignment asks for confusion matrix display and classification report; both are included separately for each question.